In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### **Dataset Overview** 🌱📊

The dataset is a comprehensive collection of real-time plant health data derived from simulated biosensors. It captures various environmental, soil, and plant-related parameters essential for assessing plant health. The dataset includes the following key features:

1. **Plant Identification** 🏷️:
   - `Plant_ID`: Unique identifier for each plant in the dataset.

2. **Soil Properties** 🌍:
   - `Soil_Moisture (%)` 💧: Indicates water content in the soil.
   - `Soil_Temperature (°C)` 🌡️: Represents temperature near the plant roots.
   - `Soil_pH` ⚗️: Reflects the acidity or alkalinity of the soil.
   - `Nitrogen_Level (mg/kg)` 🟢, `Phosphorus_Level (mg/kg)` 🟠, and `Potassium_Level (mg/kg)` 🟡: Measure nutrient levels critical for plant growth and health.

3. **Environmental Conditions** 🌤️:
   - `Ambient_Temperature (°C)` 🌡️: Temperature surrounding the plant.
   - `Humidity (%)` 💦: Air humidity levels.
   - `Light_Intensity (Lux)` ☀️: Exposure to light, crucial for photosynthesis.

4. **Plant Health Indicators** 🌿:
   - `Chlorophyll_Content (mg/m²)` 🟩: Reflects photosynthetic activity.
   - `Electrochemical_Signal (mV)` ⚡: Represents stress signals due to environmental or internal factors.

5. **Target Variable** 🎯:
   - `Plant_Health_Status`: A categorical label indicating the plant's overall health. It has three classes:
     - **Healthy** ✅: Optimal plant conditions.
     - **Moderate Stress** ⚠️: Minor deviations from ideal conditions.
     - **High Stress** ❌: Severe stress requiring immediate intervention.

The dataset spans a range of real-world conditions, offering valuable insights into the relationships between plant health and environmental, soil, and nutrient factors.

### **Objective** 🎯🤖

The primary objective of this project is to leverage the dataset to develop a robust machine learning model capable of predicting plant health status (`Healthy`, `Moderate Stress`, or `High Stress`). This involves:

1. **Data Exploration and Preprocessing** 🔍📈:
   - Understanding feature distributions, correlations, and relationships to extract meaningful insights.

2. **Model Development and Evaluation** 🛠️📊:
   - Training and evaluating multiple classification models (e.g., KNN, Decision Tree, Random Forest, Gradient Boosting) to determine the best-performing model.
   - Assessing model performance using metrics like Accuracy, AUC, Precision, Recall, and F1-Score.

3. **Insights and Recommendations** 💡🌾:
   - Providing actionable insights into factors impacting plant health.
   - Highlighting critical features (e.g., soil moisture and nitrogen levels) for targeted interventions like irrigation or fertilization.

The ultimate goal is to create a predictive tool that can assist farmers and agricultural professionals in identifying plant stress early and enabling timely corrective measures to improve crop health. 🌽🌾

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">Imports and Setup</h1>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import warnings
warnings.filterwarnings("ignore")

# <span style="color:transparent;">Load and Explore Dataset</span>

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">Load and Explore Dataset</h1>
</div>

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/plant-health-data/plant_health_data.csv')

In [ ]:
# Display basic information about the dataset
print("Shape of the dataset:", df.shape)
display(df.head())
print("\nDataset Information:")
print(df.info())
print("\nStatistical Summary:")
display(df.describe().T)

### **Insights**

#### **1. Dataset Overview:**
- **Shape of the dataset:** `(1200, 14)`
  - 1200 rows and 14 columns.
- **Preview of the dataset:** 
  - Includes features like `Soil_Moisture`, `Ambient_Temperature`, `Humidity`, `Soil_pH`, and `Plant_Health_Status`.
  - The `Timestamp` column captures the time of each measurement.

#### **2. Dataset Information:**
- **No missing values:** All columns have 1200 non-null entries.
- **Data types:**
  - **Numerical features:** 11 columns (e.g., `Soil_Moisture`, `Nitrogen_Level`, `Chlorophyll_Content`).
  - **Categorical features:** 2 columns (`Timestamp`, `Plant_Health_Status`).
  - `Timestamp` is an object and should be converted to `datetime` for time-series analysis.

#### **3. Statistical Summary:**
- **Numerical Columns Insights:**
  - `Soil_Moisture`: Ranges from 10.00% to 39.99% with a mean of 25.11%.
  - `Ambient_Temperature`: Averages around 24°C, ranging from 18°C to 30°C.
  - `Soil_pH`: Averages at 6.52, indicating neutral to slightly acidic soil.
  - `Nitrogen_Level`, `Phosphorus_Level`, and `Potassium_Level`: Centered around 30 mg/kg, with similar distributions.
  - `Chlorophyll_Content`: Indicates plant photosynthetic activity, averaging 34.75 mg/m².
  - `Electrochemical_Signal`: Mean of 0.99 mV, indicating plant stress levels.

- **Categorical Column Insights:**
  - `Plant_Health_Status`: Categorized into `Healthy`, `Moderate Stress`, and `High Stress`.
    - This column is key for classification tasks.

In [ ]:
# Check for missing and duplicated values
print(f'\nMissing values: {df.isna().sum().sum()}')
print(f'Duplicated values: {df.duplicated().sum()}')

- There are no missing values in the dataset.

- There are no duplicated rows in the dataset.

# <span style="color:transparent;">Unique Value Exploration</span>

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">Unique Value Exploration</h1>
</div>

In [ ]:
# Display the number of unique values in each column
print("\nUnique Values in Each Column:")
print(df.nunique())

In [ ]:
# Separate numerical and categorical columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
non_numerical_columns = df.select_dtypes(include=['object']).columns.tolist()

# Display the lists of numerical and categorical columns
print("\nNumerical Columns:", numerical_columns)
print("Categorical Columns:", non_numerical_columns)

In [ ]:
# Display unique values for each categorical column
for col in non_numerical_columns:
    print(f"\nColumn: {col}")
    print(f"Unique Values: {df[col].unique()}")

### **Unique Values in Each Column:**
- **Timestamp:** 1200 unique values (each entry has a unique timestamp).
- **Plant_ID:** 10 unique plants.
- **Plant_Health_Status:** 3 unique categories (`Healthy`, `Moderate Stress`, `High Stress`).
- All numerical columns (`Soil_Moisture`, `Ambient_Temperature`, etc.) have 1200 unique values, indicating no duplicates in sensor readings.

### **Column Types:**
- **Numerical Columns:**  
  `['Plant_ID', 'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 'Electrochemical_Signal']`

- **Categorical Columns:**  
  `['Timestamp', 'Plant_Health_Status']`

# <span style="color:transparent;">Exploratory Data Analysis (EDA)</span>

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">Exploratory Data Analysis (EDA)</h1>
</div>

In [ ]:
column_name = 'Plant_Health_Status'
plt.figure(figsize=(10, 4))

# First subplot: Count plot
plt.subplot(1, 2, 1)
sns.countplot(y=column_name, data=df, palette='muted')  
plt.title(f'Distribution of {column_name}')

ax = plt.gca()
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', (p.get_width(), p.get_y() + p.get_height() / 2), 
                ha='center', va='center', xytext=(10, 0), textcoords='offset points')

sns.despine(left=True, bottom=True)

# Second subplot: Pie chart
plt.subplot(1, 2, 2)
df[column_name].value_counts().plot.pie(autopct='%1.1f%%', colors=sns.color_palette('muted'), startangle=90, explode=[0.05]*df[column_name].nunique())
plt.title(f'Percentage Distribution of {column_name}')
plt.ylabel('')  

plt.tight_layout()
plt.show()

### **Insights from Plant_Health_Status**

- The bar chart reveals the count of plants in each health category:
  - **High Stress**: The largest category, indicating a significant number of plants under severe stress.
  - **Moderate Stress**: The second most frequent category.
  - **Healthy**: The least represented category, suggesting a minority of plants are in optimal condition.

- The percentage breakdown of `Plant_Health_Status` is:
  - **High Stress**: Dominates the dataset, likely due to challenging environmental or soil conditions.
  - **Moderate Stress**: Represents a transitional state between health and severe stress.
  - **Healthy**: A relatively small proportion of the dataset.

In [ ]:
# Find the earliest and latest timestamps
start_date = pd.to_datetime(df['Timestamp']).min()
end_date = pd.to_datetime(df['Timestamp']).max()

print("Start Date:", start_date)
print("End Date:", end_date)

In [ ]:
# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Check time differences between consecutive entries
time_diffs = df['Timestamp'].diff().value_counts()

print("Most common time differences:")
print(time_diffs.head(10))


### Observations from the Timestamp Analysis:

#### **1. Data Collection Period:**
- **Start Date:** October 3, 2024, at 10:54:53.
- **End Date:** November 2, 2024, at 04:54:53.
- The data was collected consistently over a span of **31 days**.

#### **2. Time Differences Between Entries:**
- The **most common time difference** between consecutive entries is **6 hours**, with 1190 occurrences. This demonstrates a well-structured schedule for data collection at 6-hour intervals.

- A few unusual time differences:
  - **-30 days +6 hours**:
    - These differences are likely due to data anomalies, errors, or overlapping timestamps mistakenly recorded as being one month apart.


In [ ]:
# Function to perform univariate analysis for numeric columns
def univariate_analysis(data, columns):
    plt.figure(figsize=(16, 24))  # Adjusted to accommodate more plots
    
    muted_colors = sns.color_palette("muted", len(columns))
    
    for i, column in enumerate(columns):
        plt.subplot(4, 3, i + 1)  
        sns.histplot(data[column], kde=True, bins=10, color=muted_colors[i])
        plt.title(f'{column.replace("_", " ")} Distribution with KDE')
        plt.xlabel(column.replace('_', ' '))
        plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

columns_to_analyze = [
    'Plant_ID', 'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 
    'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 
    'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 
    'Electrochemical_Signal'
]

# Perform univariate analysis
univariate_analysis(df, columns_to_analyze)


### Insights:

1. **Plant_ID:**
   - Uniform distribution with discrete values, as expected for an identifier.
   - All 10 plant IDs are represented equally in the dataset.

2. **Soil_Moisture:**
   - Slightly right-skewed distribution.
   - Most plants have soil moisture around the mean, with fewer instances at higher levels.

3. **Ambient_Temperature and Soil_Temperature:**
   - Both show approximately normal distributions.
   - Ambient temperature is centered around ~24°C, while soil temperature is slightly lower (approximately 20°C).

4. **Humidity:**
   - Bell-shaped distribution centered around 55%, with most values between 45% and 65%.

5. **Light_Intensity:**
   - Right-skewed distribution, indicating many instances with lower light levels and fewer high-light conditions.

6. **Soil_pH:**
   - Slightly left-skewed, with most values between 6 and 7, indicating neutral to slightly acidic soil.

7. **Nitrogen_Level, Phosphorus_Level, Potassium_Level:**
   - All three nutrients exhibit similar distributions, centered around 30 mg/kg, with slightly fewer high-nutrient instances.

8. **Chlorophyll_Content:**
   - Right-skewed distribution.
   - Most plants have chlorophyll content between 30 and 40 mg/m², indicating a typical range for photosynthesis activity.

9. **Electrochemical_Signal:**
   - Right-skewed, with a concentration around 0.9 mV.
   - Higher stress signals are rarer, suggesting that stress events are exceptional rather than common.

In [ ]:
# Select only numerical columns
numerical_df = df.select_dtypes(include=[np.number])

# Calculate skewness and kurtosis
skewness = numerical_df.skew()
kurtosis = numerical_df.kurt()

display("Skewness:", skewness)
print("\n")
display("Kurtosis:", kurtosis)

### Insights from Skewness and Kurtosis:

#### **1. Skewness:**
- **Definition:** Skewness measures the asymmetry of a distribution. A skewness near 0 indicates a symmetric distribution.
- **Observations:**
  - Most features have skewness values close to **0**, indicating that their distributions are relatively symmetric.
  - **Light_Intensity (-0.064)** and **Soil_pH (-0.085)** show a slight negative skew, suggesting a minor tail on the left side of the distribution.
  - **Chlorophyll_Content (0.052)** and **Electrochemical_Signal (0.059)** exhibit a slight positive skew, indicating a minor tail on the right.

#### **2. Kurtosis:**
- **Definition:** Kurtosis measures the "tailedness" of a distribution. A kurtosis of 3 indicates a normal distribution, while values < 3 indicate lighter tails (platykurtic), and values > 3 indicate heavier tails (leptokurtic).
- **Observations:**
  - All features have kurtosis values significantly below **3** (e.g., ~ -1.2), suggesting **platykurtic distributions**. This means the features generally have lighter tails and a flatter peak than a normal distribution.
  - Features like **Phosphorus_Level (-1.16)** and **Electrochemical_Signal (-1.17)** show the least deviation, while **Humidity (-1.24)** and **Plant_ID (-1.22)** are slightly flatter.

In [ ]:
# Function to perform univariate analysis for numeric columns with boxplots and statistics
def univariate_analysis(data, column, title):
    plt.figure(figsize=(10, 2))
    
    color = sns.color_palette("muted")[columns_to_analyze.index(column) % len(sns.color_palette("muted"))]
    
    sns.boxplot(x=data[column], color=color)
    plt.title(f'{title} Boxplot')
    
    plt.tight_layout()
    plt.show()

    print(f'\nSummary Statistics for {title}:\n', data[column].describe())

columns_to_analyze = [
    'Plant_ID', 'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 
    'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 
    'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 
    'Electrochemical_Signal'
]

# Iterate through columns and perform univariate analysis
for column in columns_to_analyze:
    univariate_analysis(df, column, column.replace('_', ' '))


### Insights:

1. **Plant_ID:**
- Uniform distribution across all plant IDs.
- No outliers as expected since it's an identifier.

2. **Soil_Moisture:**
- Range: 10.0% to 40.0%.
- Outliers observed at both low and high ends, possibly indicating waterlogged or dehydrated plants.

3. **Ambient_Temperature and `Soil_Temperature:**
- Ambient temperature: 18°C to 30°C, centered around ~24°C.
- Soil temperature: Slightly lower than ambient, ranging from 15°C to 25°C.
- No significant outliers, indicating stable temperature conditions.

4. **Humidity:**
- Range: 40% to ~70%, centered around 55%.
- A small spread with minimal outliers, suggesting controlled environmental humidity.

5. **Light_Intensity:**
- Range: 200 to 1000 Lux.
- Outliers observed in lower light conditions (<300 Lux), which might indicate plants in shaded or low-light environments.

6. **Soil_pH:**
- Range: 5.5 to 7.5, indicating neutral to slightly acidic soil.
- Minimal outliers, reflecting consistent soil conditions across samples.

7. Nutrient Levels (`Nitrogen`, `Phosphorus`, `Potassium`):**
- Range: 10 to 50 mg/kg for all nutrients.
- Outliers observed at both extremes, suggesting variability in soil fertility.

8. **Chlorophyll_Content:**
- Range: 20 to 50 mg/m², centered around 34 mg/m².
- Outliers at higher chlorophyll levels (>45 mg/m²), indicating exceptional photosynthetic activity.

9. **Electrochemical_Signal:**
- Range: 0.002 to ~2.0 mV, centered around 0.98 mV.
- Outliers on the higher end (>1.5 mV) likely indicate stress conditions.



Features like `Soil_Moisture`, `Light_Intensity`, and nutrient levels have significant variability and outliers, which could affect plant health.
Stable variables like temperature, humidity, and soil pH show minimal variability, reflecting controlled conditions.


### **Summary of Categorization with Descriptions:**

| **Category**               | **Features**                                                                 | **Description**                                                                                     |
|-----------------------------|------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------|
| **Plant Identification**    | `Plant_ID`                                                                  | Unique identifier for each plant. Used to group data specific to individual plants.                 |
| **Soil Properties**          | `Soil_Moisture`, `Soil_Temperature`, `Soil_pH`, `Nitrogen_Level`, `Phosphorus_Level`, `Potassium_Level` | Physical and chemical characteristics of the soil, crucial for plant hydration, nutrition, and root health. |
| **Environmental Conditions** | `Ambient_Temperature`, `Humidity`, `Light_Intensity`                       | Surrounding environmental factors that influence plant growth and stress levels.                    |
| **Plant Health Indicators**  | `Chlorophyll_Content`, `Electrochemical_Signal`                            | Direct indicators of plant health and stress, reflecting photosynthesis efficiency and stress response. |


In [ ]:
# Convert the Timestamp column to datetime format if not already
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Create a 'Week' column to group by week
df['Week'] = df['Timestamp'].dt.to_period('W').apply(lambda r: r.start_time)

# Aggregate Plant Health Status weekly for each Plant ID
weekly_health_status = (
    df.groupby(['Plant_ID', 'Week', 'Plant_Health_Status'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

weekly_health_status.columns.name = None
weekly_health_status = weekly_health_status.rename(columns={
    'High Stress': 'High_Stress_Count',
    'Moderate Stress': 'Moderate_Stress_Count',
    'Healthy': 'Healthy_Count'
})

weekly_health_status

In [ ]:
# Compare Plant_Health_Status with Soil Properties
# Define soil properties
soil_properties = [
    'Soil_Moisture', 'Soil_Temperature', 'Soil_pH', 
    'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level'
]

# Create subplots to visualize the relationship between Plant_Health_Status and soil properties
plt.figure(figsize=(16, 20))
for i, feature in enumerate(soil_properties):
    plt.subplot(3, 2, i + 1)
    sns.boxplot(x='Plant_Health_Status', y=feature, data=df, palette='muted')
    plt.title(f'{feature} vs Plant Health Status')
    plt.xlabel('Plant Health Status')
    plt.ylabel(feature.replace('_', ' '))
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


### Insights: Comparison of Plant_Health_Status with Soil Properties

1. **`Soil_Moisture` vs `Plant_Health_Status`:**
   - **Healthy plants**: Tend to have moderate soil moisture levels, suggesting optimal hydration.
   - **High Stress plants**: Exhibit a wider range of soil moisture, with extreme low or high values, potentially indicating under-watering or over-watering.

2. **`Soil_Temperature` vs `Plant_Health_Status`:**
   - **Healthy plants**: Soil temperature is consistent, generally within a moderate range (~18°C to 22°C).
   - **High Stress plants**: Wider variability, with some instances of unusually low or high temperatures, likely affecting root function.

3. **`Soil_pH` vs `Plant_Health_Status`:**
   - **Healthy plants**: Prefer soil pH near neutral (6.5–7.0), which is optimal for nutrient availability.
   - **High Stress plants**: Soil pH shows more extreme acidic or alkaline values, possibly limiting nutrient uptake.

4. **`Nitrogen_Level` vs `Plant_Health_Status`:**
   - **Healthy plants**: Higher nitrogen levels support better leaf growth and overall plant health.
   - **High Stress plants**: Generally associated with lower nitrogen levels, which may stunt growth.

5. **`Phosphorus_Level` vs `Plant_Health_Status`:**
   - **Healthy plants**: Moderate phosphorus levels support healthy root development.
   - **High Stress plants**: Show wider variability, with some instances of low phosphorus, potentially impairing root and flower development.

6. **`Potassium_Level` vs `Plant_Health_Status`:**
   - **Healthy plants**: Moderate to high potassium levels provide resilience to stress and diseases.
   - **High Stress plants**: Tend to have lower potassium levels, which may compromise disease resistance.

### Key observations:
- Soil properties like **pH, nitrogen, and potassium levels** are critical for maintaining plant health.
- Extreme values in soil moisture and pH are closely linked to high-stress conditions.
- Healthy plants thrive within moderate ranges of most soil properties, while variability or extremes correlate with plant stress.

In [ ]:
# Compare Plant_Health_Status with Environmental Conditions
# Define environmental condition features
environmental_conditions = [
    'Ambient_Temperature', 'Humidity', 'Light_Intensity'
]

# Create subplots to visualize the relationship between Plant_Health_Status and environmental conditions
plt.figure(figsize=(16, 12))
for i, feature in enumerate(environmental_conditions):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(x='Plant_Health_Status', y=feature, data=df, palette='muted')
    plt.title(f'{feature} vs Plant Health Status')
    plt.xlabel('Plant Health Status')
    plt.ylabel(feature.replace('_', ' '))
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


### Insights: Comparison of Plant_Health_Status with Environmental Conditions

1. **`Ambient_Temperature` vs `Plant_Health_Status`:**
   - **Healthy plants**: Tend to thrive in a moderate temperature range (around ~24°C), which is optimal for photosynthesis and growth.
   - **High Stress plants**: Exhibit more variability in ambient temperature, with some instances of higher or lower extremes, which could stress plants.

2. **`Humidity` vs `Plant_Health_Status`:**
   - **Healthy plants**: Prefer moderate humidity levels (~55%), ensuring efficient transpiration and water transport.
   - **High Stress plants**: Show wider variability, with instances of both low and high humidity, potentially impacting water loss and stress responses.

3. **`Light_Intensity` vs `Plant_Health_Status`:**
   - **Healthy plants**: Tend to receive moderate to high light intensity (around 600–800 Lux), crucial for sufficient photosynthesis.
   - **High Stress plants**: Experience greater variability in light intensity, with some cases of low light (likely limiting photosynthesis) or very high light (possibly causing photooxidative stress).

### Key observations:
- **Moderate environmental conditions** (temperature, humidity, and light intensity) are closely associated with plant health.
- **Extremes or variability** in these conditions often correlate with higher stress levels, suggesting plants are more vulnerable to environmental fluctuations.
- **Light intensity** appears to be a critical factor, with healthy plants consistently exposed to optimal lighting, while stressed plants face irregular or inadequate exposure.

In [ ]:
# Compare Plant_Health_Status with Plant Health Indicators

# Define plant health indicator features
health_indicators = [
    'Chlorophyll_Content', 'Electrochemical_Signal'
]

# Create subplots to visualize the relationship between Plant_Health_Status and plant health indicators
plt.figure(figsize=(16, 8))
for i, feature in enumerate(health_indicators):
    plt.subplot(1, 2, i + 1)
    sns.boxplot(x='Plant_Health_Status', y=feature, data=df, palette='muted')
    plt.title(f'{feature} vs Plant Health Status')
    plt.xlabel('Plant Health Status')
    plt.ylabel(feature.replace('_', ' '))
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


### Insights: Comparison of Plant_Health_Status with Plant Health Indicators

1. **`Chlorophyll_Content` vs `Plant_Health_Status`:**
   - **Healthy plants**: Have the highest chlorophyll content, averaging around 40–45 mg/m². This indicates robust photosynthetic activity.
   - **Moderate Stress plants**: Show reduced chlorophyll levels, with values around 30–35 mg/m².
   - **High Stress plants**: Exhibit the lowest chlorophyll content (~20–30 mg/m²), suggesting compromised photosynthesis and plant health.

2. **`Electrochemical_Signal` vs `Plant_Health_Status`:**
   - **Healthy plants**: Have the lowest electrochemical signal values, indicating minimal stress responses.
   - **Moderate Stress plants**: Show moderately higher signal values, reflecting some stress response.
   - **High Stress plants**: Exhibit the highest electrochemical signals (~1.5–2.0 mV), indicating strong stress responses due to adverse conditions.

### Key observations:
- **Chlorophyll content** serves as a direct indicator of plant photosynthetic efficiency and overall health. Lower levels are strongly associated with stress conditions.
- **Electrochemical signal** is a robust marker for plant stress, with higher values reflecting greater stress levels.
- The inverse relationship between chlorophyll content and electrochemical signal underscores their complementary roles in assessing plant health.

In [ ]:
# Compare Plant_Health_Status with Plant Health Indicators vs Soil Properties using scatter plots

# Define soil properties and plant health indicators
soil_properties = [
    'Soil_Moisture', 'Soil_Temperature', 'Soil_pH', 
    'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level'
]

health_indicators = [
    'Chlorophyll_Content', 'Electrochemical_Signal'
]

# Create scatter plots for each combination of plant health indicators and soil properties
plt.figure(figsize=(30, 15))
plot_index = 1

for health_indicator in health_indicators:
    for soil_property in soil_properties:
        plt.subplot(len(health_indicators), len(soil_properties), plot_index)
        sns.scatterplot(
            x=soil_property, 
            y=health_indicator, 
            hue='Plant_Health_Status', 
            data=df, 
            palette='muted', 
            alpha=0.7
        )
        plt.title(f'{health_indicator} vs {soil_property}')
        plt.xlabel(soil_property.replace('_', ' '))
        plt.ylabel(health_indicator.replace('_', ' '))
        plt.legend(title='Health Status', loc='upper right')
        plot_index += 1

plt.tight_layout()
plt.show()


### Insights: Plant Health Indicators vs Soil Properties 

1. **`Chlorophyll_Content` vs Soil Properties:**
   - **Soil Moisture:** Chlorophyll content is higher in healthy plants with moderate soil moisture. Extremely low or high soil moisture correlates with reduced chlorophyll content in stressed plants.
   - **Soil pH:** Healthy plants have a balanced pH near neutral, which maximizes chlorophyll production. Stress increases with more acidic or alkaline soil conditions.
   - **Nutrient Levels (Nitrogen, Phosphorus, Potassium):** Healthy plants exhibit higher nutrient levels, supporting better chlorophyll content. Stress is associated with nutrient deficiencies.

2. **`Electrochemical_Signal` vs Soil Properties:**
   - **Soil Moisture:** Electrochemical signal values increase (indicating stress) with extreme moisture levels, suggesting over-watering or dehydration impacts stress responses.
   - **Soil Temperature:** Moderate soil temperature correlates with lower stress levels. Stressed plants are observed with deviations from this range.
   - **Soil pH:** High stress (elevated electrochemical signals) is seen in plants with extreme pH values, possibly due to limited nutrient availability.
   - **Nutrient Levels:** Low nutrient levels (especially nitrogen) are associated with high electrochemical signal values, indicating nutrient stress.

### Key Observations:
- **Chlorophyll Content:**
  - Directly reflects plant health. Higher chlorophyll levels in healthy plants correlate with optimal soil conditions.
- **Electrochemical Signal:**
  - Serves as a stress marker. Higher values are linked to extreme soil conditions (moisture, pH, nutrient levels) and plant health deterioration.

In [ ]:
# Compare Plant_Health_Status with Plant Health Indicators vs Environmental Conditions using scatter plots

# Define environmental conditions and plant health indicators
environmental_conditions = [
    'Ambient_Temperature', 'Humidity', 'Light_Intensity'
]

health_indicators = [
    'Chlorophyll_Content', 'Electrochemical_Signal'
]

# Create scatter plots for each combination of plant health indicators and environmental conditions
plt.figure(figsize=(20, 12))
plot_index = 1

for health_indicator in health_indicators:
    for env_condition in environmental_conditions:
        plt.subplot(len(health_indicators), len(environmental_conditions), plot_index)
        sns.scatterplot(
            x=env_condition, 
            y=health_indicator, 
            hue='Plant_Health_Status', 
            data=df, 
            palette='muted', 
            alpha=0.7
        )
        plt.title(f'{health_indicator} vs {env_condition}')
        plt.xlabel(env_condition.replace('_', ' '))
        plt.ylabel(health_indicator.replace('_', ' '))
        plt.legend(title='Health Status', loc='upper right')
        plot_index += 1

plt.tight_layout()
plt.show()


In [ ]:
# Explore relationships between Plant_Health_Status, Plant Health Indicators, and Environmental Conditions

# Group data by Plant_Health_Status and calculate mean values
status_analysis_mean = df.groupby('Plant_Health_Status')[
    ['Chlorophyll_Content', 'Electrochemical_Signal', 'Ambient_Temperature', 'Humidity', 'Light_Intensity']
].mean()

# Group data by Plant_Health_Status and calculate standard deviation
status_analysis_std = df.groupby('Plant_Health_Status')[
    ['Chlorophyll_Content', 'Electrochemical_Signal', 'Ambient_Temperature', 'Humidity', 'Light_Intensity']
].std()

# Displaying the mean values table
print("----- Mean Values by Plant Health Status -----")
display(status_analysis_mean)

# Displaying the standard deviation table
print("\n----- Standard Deviation by Plant Health Status -----")
display(status_analysis_std)

### **Insights from the Mean Table:**

1. **Chlorophyll Content:**
   - Healthy plants exhibit the highest average chlorophyll content (approximately 34.97 mg/m²), indicating optimal photosynthetic activity.
   - Moderate and High Stress plants show slightly lower average chlorophyll levels (approximately 34.57–34.76 mg/m²), reflecting reduced photosynthesis under stress.

2. **Electrochemical Signal:**
   - Healthy plants have the lowest average electrochemical signal (approximately 0.97 mV), suggesting minimal stress.
   - High Stress plants show a slightly elevated signal (approximately 0.99 mV), indicating stronger stress responses.

3. **Ambient Temperature:**
   - The mean temperature remains consistent across all health statuses (approximately 24°C), suggesting similar environmental temperature conditions for all plants.

4. **Humidity:**
   - Healthy plants are associated with slightly higher humidity (approximately 55.36%) compared to High Stress plants (approximately 54.40%), which could indicate optimal transpiration levels in healthy plants.

5. **Light Intensity:**
   - High Stress plants experience slightly higher light intensity (approximately 618.67 Lux) compared to Healthy plants (approximately 603.46 Lux). Excessive light might contribute to stress conditions.

### **Insights from the Standard Deviation Table:**

1. **Chlorophyll Content:**
   - Similar variability (approximately 8.7 mg/m²) is observed across all health statuses, indicating consistent differences in photosynthetic activity.

2. **Electrochemical Signal:**
   - High Stress plants exhibit slightly higher variability (approximately 0.59 mV) compared to Healthy and Moderate Stress plants (approximately 0.56 mV). This could reflect greater fluctuations in stress response.

3. **Ambient Temperature:**
   - Healthy plants show slightly more variability in temperature (approximately 3.54°C), potentially indicating exposure to a wider range of conditions compared to stressed plants.

4. **Humidity:**
   - Variability in humidity (approximately 8.7–8.8%) is consistent across health statuses, reflecting uniform environmental conditions.

5. **Light Intensity:**
   - The standard deviation of light intensity (approximately 225–230 Lux) is nearly identical for all health statuses, suggesting similar fluctuations in light exposure.

### **Key Observation:**

- **Chlorophyll Content and Electrochemical Signal**:
  - Healthy plants demonstrate higher chlorophyll and lower stress signals, directly reflecting better health.
  - High Stress plants exhibit reduced chlorophyll and slightly more variable stress signals, indicating adverse conditions.

- **Environmental Conditions**:
  - Consistent mean and variability in temperature and humidity suggest a controlled environment, but subtle differences in light intensity highlight its potential impact on plant stress.

In [ ]:
# Explore relationships between Plant_Health_Status and Soil Properties

# Group data by Plant_Health_Status and calculate mean values for soil properties
soil_properties = [
    'Soil_Moisture', 'Soil_Temperature', 'Soil_pH', 
    'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level'
]

soil_mean = df.groupby('Plant_Health_Status')[soil_properties].mean()
soil_std = df.groupby('Plant_Health_Status')[soil_properties].std()

# Displaying the mean values table
print("----- Mean Soil Properties by Plant Health Status -----")
display(soil_mean)

# Displaying the standard deviation table
print("\n----- Standard Deviation of Soil Properties by Plant Health Status -----")
display(soil_std)

### **Key observations:**
1. **Soil Moisture:** 
   - Directly correlates with plant health. High Stress plants face severe variability and lower average moisture, while Healthy plants enjoy consistent hydration.

2. **Soil pH:**
   - Near-neutral pH supports Healthy and Moderate Stress plants. Slight deviations in High Stress plants may disrupt nutrient uptake.

3. **Nutrient Levels:**
   - Nitrogen, phosphorus, and potassium are highest in Healthy plants, promoting robust growth and resilience.
   - High Stress plants consistently show lower mean nutrient levels and greater variability, reflecting suboptimal conditions.

This analysis highlights the importance of balanced soil properties for maintaining plant health and reducing stress levels.

In [ ]:
# Define nutrient levels
nutrient_levels = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']

# Group data by Plant_ID and calculate mean nutrient levels
nutrient_mean = df.groupby('Plant_ID')[nutrient_levels].mean()

# Group data by Plant_ID and calculate standard deviation for nutrient levels
nutrient_std = df.groupby('Plant_ID')[nutrient_levels].std()

# Displaying the mean values table
print("------ Mean Nutrient Levels by Plant ID ------")
display(nutrient_mean)

# Displaying the standard deviation table
print("\n------ Standard Deviation of Nutrient Levels by Plant ID ------")
display(nutrient_std)

### Insights: Nutrient Levels by Plant ID

#### **Mean Nutrient Levels by Plant ID**
1. **Nitrogen Level:**
   - Consistently higher for some plants, indicating better soil nitrogen availability for these plants.
   - Lower nitrogen levels for certain Plant IDs might suggest localized nutrient deficiencies.

2. **Phosphorus Level:**
   - Average phosphorus levels are relatively uniform across Plant IDs, though some variations hint at soil differences or plant-specific uptake rates.

3. **Potassium Level:**
   - Variability across Plant IDs indicates differing soil potassium availability or plant-specific nutrien demands.


#### **Standard Deviation of Nutrient Levels by Plant ID**
1. **Nitrogen Level:**
   - High variability for certain plants suggests inconsistent soil nitrogen distribution or varying uptake efficiency.
   - Lower variability indicates more uniform nutrient conditions.

2. **Phosphorus and Potassium Levels:**
   - Similar trends as nitrogen, with variability pointing to localized nutrient imbalances or plant-specific factors.


In [ ]:
# Define nutrient levels
nutrients = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']

# Set up the plot
plt.figure(figsize=(18, 10))

for i, nutrient in enumerate(nutrients):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(x='Plant_ID', y=nutrient, data=df, palette='muted')
    plt.title(f'{nutrient.replace("_", " ")} by Plant ID')
    plt.xlabel('Plant ID')
    plt.ylabel(nutrient.replace('_', ' '))
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


#### **Insights:**
- Nutrient levels exhibit variability both within and across Plant IDs.
- Outliers indicate potential extremes in soil conditions or measurement anomalies.
- Certain Plant IDs consistently have higher or lower nutrient levels, reflecting soil heterogeneity or plant-specific nutrient needs.

These observations highlight the importance of tailored nutrient management to address plant-specific deficiencies and optimize growth.

In [ ]:
# Calculate mean nutrient levels by Plant ID and Plant Health Status
nutrient_status_mean = df.groupby(['Plant_ID', 'Plant_Health_Status'])[
    ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']
].mean().reset_index()

# Separate the nutrient status mean table into three based on Plant_Health_Status
nutrient_high_stress = nutrient_status_mean[nutrient_status_mean['Plant_Health_Status'] == 'High Stress']
nutrient_moderate_stress = nutrient_status_mean[nutrient_status_mean['Plant_Health_Status'] == 'Moderate Stress']
nutrient_healthy = nutrient_status_mean[nutrient_status_mean['Plant_Health_Status'] == 'Healthy']

# Display the three tables to the user
print("----- Nutrient Levels for High Stress Plants -----")
display(nutrient_high_stress)

print("\n----- Nutrient Levels for Moderate Stress Plants -----")
display(nutrient_moderate_stress)

print("\n----- Nutrient Levels for Healthy Plants -----")
display(nutrient_healthy)

### Key Observations:
1. **Healthy Plants:**
   - Tend to have balanced and higher nutrient levels across all Plant IDs.
   - Consistency in nitrogen, phosphorus, and potassium suggests optimal soil conditions for these plants.

2. **Moderate Stress Plants:**
   - Show slight reductions in nutrient levels compared to healthy plants.
   - Variability in phosphorus and potassium levels may contribute to moderate stress conditions.

3. **High Stress Plants:**
   - Exhibit significantly lower nutrient levels for many Plant IDs, particularly nitrogen and phosphorus.
   - Indicates nutrient deficiencies are likely contributors to high-stress conditions.

In [ ]:
# Visualize the nutrient levels across different Plant_Health_Status categories (High Stress, Moderate Stress, Healthy) for each Plant_ID
# Define nutrient levels
nutrients = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']

# Set up the plot
plt.figure(figsize=(18, 12))
for i, nutrient in enumerate(nutrients):
    plt.subplot(2, 2, i + 1)
    sns.barplot(
        x='Plant_ID', 
        y=nutrient, 
        hue='Plant_Health_Status', 
        data=nutrient_status_mean, 
        palette='muted'
    )
    plt.title(f'{nutrient.replace("_", " ")} by Plant ID and Health Status')
    plt.xlabel('Plant ID')
    plt.ylabel(nutrient.replace('_', ' '))
    plt.legend(title='Health Status', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# Environmental Conditions vs Plant_Health_Status using FacetGrid with improved styling

# Define environmental conditions to plot
environmental_conditions = ['Ambient_Temperature', 'Humidity', 'Light_Intensity']

# Create FacetGrid plots for each environmental condition with improved styling
for condition in environmental_conditions:
    g = sns.FacetGrid(
        df, 
        col='Plant_Health_Status', 
        height=4, 
        aspect=1.2, 
        sharex=True, 
        sharey=True, 
    )
    g.map(sns.histplot, condition, kde=True, bins=10, color=None)
    g.set_axis_labels(condition.replace('_', ' '), 'Frequency')
    g.set_titles("{col_name} - Plant Health Status")
    g.fig.subplots_adjust(top=0.85)  
    g.fig.suptitle(
        f'Distribution of {condition.replace("_", " ")} by Plant Health Status', 
        fontsize=16, 
        y=0.98  
    )
    plt.show()


### Insights: Environmental Conditions vs Plant_Health_Status 

#### **1. Ambient Temperature:**
- **Healthy Plants:**
  - The distribution is centered around ~24°C, with a narrow range indicating consistent optimal conditions.
- **Moderate Stress:**
  - Slightly wider distribution, with values around 22–26°C, suggesting tolerance to minor deviations.
- **High Stress:**
  - The widest range, with some instances below 20°C or above 28°C, indicating extreme conditions leading to stress.

#### **2. Humidity:**
- **Healthy Plants:**
  - Centered around 55%, with a narrow range (~50–60%), reflecting ideal transpiration conditions.
- **Moderate Stress:**
  - Slightly lower peak, with a wider spread (~45–65%), indicating some impact of humidity on stress levels.
- **High Stress:**
  - Broad distribution, with peaks at both lower (<45%) and higher (>65%) humidity levels, showing significant deviations from optimal conditions.

#### **3. Light Intensity:**
- **Healthy Plants:**
  - Consistently higher light intensity (~600–800 Lux), supporting photosynthesis.
- **Moderate Stress:**
  - Slightly lower light levels, with some overlap into suboptimal regions (<600 Lux).
- **High Stress:**
  - Broad distribution, with peaks at both very low (<400 Lux) and very high (>900 Lux) intensities, reflecting light-related stress.


### Key observations:
- **Healthy Plants** thrive under moderate and consistent environmental conditions.
- **Stress Levels** increase with deviations from optimal temperature, humidity, and light intensity.
- The **broader distributions for High Stress plants** highlight the impact of environmental extremes on plant health.

In [ ]:
# Define plant health indicators to plot
health_indicators = ['Chlorophyll_Content', 'Electrochemical_Signal']

# Create FacetGrid plots for each health indicator
for indicator in health_indicators:
    g = sns.FacetGrid(
        df, 
        col='Plant_Health_Status', 
        height=4, 
        aspect=1.2, 
        sharex=True, 
        sharey=True
    )
    g.map(sns.histplot, indicator, kde=True, bins=10, color=None)
    g.set_axis_labels(indicator.replace('_', ' '), 'Frequency')
    g.set_titles("{col_name} - Plant Health Status")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle(
        f'Distribution of {indicator.replace("_", " ")} by Plant Health Status', 
        fontsize=16, 
        y=0.98
    )
    plt.show()


### Insights: Plant Health Indicators vs Plant_Health_Status 

#### **1. Chlorophyll Content:**
- **Healthy Plants:**
  - Distribution peaks at higher chlorophyll levels (~40–45 mg/m²), indicating robust photosynthetic activity.
- **Moderate Stress Plants:**
  - Chlorophyll levels are slightly lower (~30–35 mg/m²), suggesting reduced photosynthetic efficiency due to mild stress.
- **High Stress Plants:**
  - Distribution shifts further down (~20–30 mg/m²), reflecting significant photosynthetic impairment under high-stress conditions.

#### **2. Electrochemical Signal:**
- **Healthy Plants:**
  - Stress signals are minimal, with most values clustering around ~0.8–1.0 mV.
- **Moderate Stress Plants:**
  - Slightly elevated stress signals (~1.0–1.2 mV), reflecting moderate environmental or physiological stress.
- **High Stress Plants:**
  - Higher electrochemical signals (~1.2–1.5 mV), indicating strong stress responses due to adverse conditions.


### Key observations:
- **Chlorophyll Content:** Strongly correlates with plant health. Higher levels are indicative of healthy photosynthesis, while reductions signal increasing stress.
- **Electrochemical Signal:** Provides a direct measure of plant stress, with values rising as health deteriorates.

In [ ]:
# Soil Properties vs Plant_Health_Status using FacetGrid

# Define soil properties
soil_properties = [
    'Soil_Moisture', 'Soil_Temperature', 'Soil_pH', 
    'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level'
]

# Create FacetGrid plots for each soil property
for property in soil_properties:
    g = sns.FacetGrid(
        df, 
        col='Plant_Health_Status', 
        height=4, 
        aspect=1.2, 
        sharex=True, 
        sharey=True
    )
    g.map(sns.histplot, property, kde=True, bins=10, color=None)
    g.set_axis_labels(property.replace('_', ' '), 'Frequency')
    g.set_titles("{col_name} - Plant Health Status")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle(
        f'Distribution of {property.replace("_", " ")} by Plant Health Status', 
        fontsize=16, 
        y=0.98
    )
    plt.show()


### Insights: Soil Properties vs Plant_Health_Status 

#### **1. Soil Moisture:**
- **Healthy Plants:**
  - Distribution peaks at higher moisture levels (~30–35%), reflecting adequate hydration.
- **Moderate Stress Plants:**
  - Distribution shifts to lower soil moisture (~20–25%), indicating reduced hydration levels.
- **High Stress Plants:**
  - The lowest soil moisture levels (~15–20%) are most common, signaling water-related stress.

#### **2. Soil Temperature:**
- **Healthy Plants:**
  - Distribution is centered around ~20°C, representing optimal soil conditions for root activity.
- **Moderate Stress Plants:**
  - Similar distribution as healthy plants but with slightly more variability.
- **High Stress Plants:**
  - Peaks slightly below ~20°C, with broader variability indicating temperature-related stress.

#### **3. Soil pH:**
- **Healthy Plants:**
  - Distribution is centered around neutral pH (~6.5–7.0), ensuring nutrient availability.
- **Moderate Stress Plants:**
  - Slight shift in distribution with more instances near suboptimal pH (~6.0–6.5).
- **High Stress Plants:**
  - Distribution shows greater variability, with some values deviating from neutral, suggesting pH imbalance.

#### **4. Nitrogen Level:**
- **High Stress:**
  - Most plants have nitrogen levels below 25 mg/kg, indicating significant deficiencies.
  - A small fraction of plants shows nitrogen levels above 40 mg/kg, which might suggest variability in nitrogen availability.
- **Moderate Stress:**
  - Nitrogen levels are more evenly distributed, with most values around 25–35 mg/kg.
- **Healthy:**
  - Higher nitrogen levels are observed, clustering around 35–45 mg/kg, supporting plant growth and resilience.

#### **5. Phosphorus Level:**
- **High Stress:**
  - Plants exhibit a broad range of phosphorus levels, with peaks around 25 mg/kg, reflecting deficiencies in some cases.
- **Moderate Stress:**
  - Phosphorus levels are moderately distributed (~30–40 mg/kg), showing improved conditions compared to high stress.
- **Healthy:**
  - Higher phosphorus levels are consistent (~35–45 mg/kg), indicating adequate nutrient supply for healthy root development.

#### **6. Potassium Level:**
- **High Stress:**
  - A broader range with a peak near 25–30 mg/kg, indicating variability in potassium availability.
- **Moderate Stress:**
  - Levels are more consistent (~30–40 mg/kg), improving plant resilience.
- **Healthy:**
  - Potassium levels are higher and more stable (~35–45 mg/kg), aiding in disease resistance.

### Key observations:
- **Soil Moisture:** Closely linked to plant health, with high-stress conditions strongly associated with low moisture levels.
- **Soil Temperature:** Healthy plants thrive in stable temperatures, while stressed plants face more variability.
- **Soil pH:** Healthy plants require neutral pH, while deviations contribute to stress.
- **Nutrient Levels:**
   - Nitrogen, phosphorus, and potassium deficiencies are strongly linked to plant stress.
   - Healthy plants consistently show higher and more stable nutrient levels.

In [ ]:
numerical_features = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_features].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix, 
    annot=True, 
    fmt='.2f', 
    cmap='YlGnBu', 
    square=True, 
    linewidths=0.5, 
    cbar_kws={"shrink": 0.8}
)
plt.title('Correlation Heatmap for Numerical Features', fontsize=16)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


### Detailed Insights from Correlation Table:

### **1. Strongest Positive Correlations:**
- **Chlorophyll_Content and Soil_Temperature** (~0.05):
  - This weak but positive correlation indicates that moderate soil temperatures may slightly enhance chlorophyll content, supporting photosynthesis.
  
- **Soil_Moisture and Humidity** (~0.054):
  - A minor positive relationship exists between soil moisture and air humidity, likely due to the effect of soil hydration on the surrounding microenvironment.

### **2. Strongest Negative Correlations:**
- **Nitrogen_Level and Chlorophyll_Content** (~-0.045):
  - Surprisingly, this negative correlation suggests that higher nitrogen levels might not directly contribute to chlorophyll production, potentially due to other limiting factors.
  
- **Phosphorus_Level and Soil_Temperature** (~-0.040):
  - High phosphorus levels tend to coincide with lower soil temperatures, which could be related to specific soil conditions or plant uptake dynamics.

### **3. Relationships Between Soil Properties:**
- **Soil_pH and Soil_Temperature** (~-0.025):
  - Soil pH slightly decreases with increasing soil temperature, hinting at complex soil chemistry interactions.

- **Soil_Moisture and Soil_pH** (~-0.044):
  - A weak negative correlation indicates that higher soil moisture might marginally lower soil pH.

### **4. Plant Health Indicators:**
- **Electrochemical_Signal and Chlorophyll_Content** (~0.025):
  - A weak positive correlation suggests that stress signals (electrochemical activity) may not always align with reduced photosynthetic activity.

- **Electrochemical_Signal and Soil_Temperature** (~-0.024):
  - Stress signals slightly increase with decreasing soil temperature, which may indicate suboptimal conditions for root activity.

### **5. Environmental Conditions:**
- **Ambient_Temperature and Light_Intensity** (~-0.025):
  - A weak inverse relationship suggests that high light intensity might not coincide with warmer temperatures, possibly due to shade or time-of-day effects.

- **Humidity and Light_Intensity** (~-0.023):
  - A small negative correlation reflects reduced humidity levels under intense light, likely due to increased evapotranspiration.

### **Key observesions:**
- Most correlations are weak, highlighting the complex and independent nature of these features.
- Relationships between **soil properties** and **plant health indicators** are minimal, suggesting other external factors might influence plant health.
- Negative correlations between nutrients (e.g., nitrogen) and chlorophyll content suggest that nutrient levels alone do not guarantee higher photosynthesis; other environmental factors also play a role.

In [ ]:
# Convert Timestamp to datetime format and set it as the index
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)

# Define features to analyze
time_series_features = [
    'Soil_Moisture', 'Soil_Temperature', 'Ambient_Temperature', 
    'Humidity', 'Soil_pH', 'Light_Intensity', 'Nitrogen_Level', 
    'Phosphorus_Level', 'Potassium_Level'
]

# Resample data for daily and weekly averages
daily_averages = df[time_series_features].resample('D').mean()
weekly_averages = df[time_series_features].resample('W').mean()


In [ ]:
# Plot daily trends for selected features
plt.figure(figsize=(15, 10))
for i, feature in enumerate(time_series_features[:5]):  
    plt.subplot(3, 2, i + 1)
    daily_averages[feature].plot(
        title=f'Daily Trend of {feature.replace("_", " ")}', 
        ylabel=feature.replace('_', ' '), 
        xlabel='Date'
    )
plt.tight_layout()
plt.show()

### Insights from the Daily Trends of Selected Features:

#### **1. Soil Moisture:**
- The soil moisture levels show daily fluctuations, with values generally ranging between 22% and 28%.
- There are clear peaks and troughs, indicating periodic watering or environmental changes that influence moisture retention.
- A slight downward trend is observed mid-month, possibly due to reduced watering or higher evaporation rates.

#### **2. Soil Temperature:**
- The soil temperature remains relatively stable, with minor variations around 19°C to 21°C.
- The consistency reflects minimal external disruptions, possibly due to controlled environmental conditions or stable weather patterns.

#### **3. Ambient Temperature:**
- Ambient temperature fluctuates daily, with values ranging between 22.5°C and 25°C.
- The trend shows periodic peaks and drops, likely corresponding to natural day-night or seasonal temperature cycles.

#### **4. Humidity:**
- Humidity exhibits noticeable variability, ranging between 52% and 58%.
- There is a peak in humidity mid-month, which might be due to weather conditions or increased watering practices.
- A sharp drop near the end of the month indicates a potential dry spell or environmental shift.

#### **5. Soil pH:**
- The soil pH fluctuates within a narrow range of 6.4 to 6.7, reflecting overall stability in soil acidity.
- There are minor daily variations, possibly influenced by watering, fertilization, or leaching processes.

In [ ]:
# Plot weekly trends for the remaining 4 features
plt.figure(figsize=(15, 10))
remaining_features = time_series_features[5:]  

for i, feature in enumerate(remaining_features):
    plt.subplot(2, 2, i + 1)
    weekly_averages[feature].plot(
        title=f'Weekly Trend of {feature.replace("_", " ")}', 
        ylabel=feature.replace('_', ' '), 
        xlabel='Date'
    )
plt.tight_layout()
plt.show()


### Insights from Weekly Trends of the Remaining Features:

#### **1. Light Intensity:**
- The weekly trend shows a peak in light intensity during the second week (approximately 625 Lux), followed by a dip in the third week and a recovery in the fourth week.
- This fluctuation may be due to varying weather conditions (e.g., cloudy vs. sunny days) or changes in artificial lighting setups.

#### **2. Nitrogen Level:**
- Nitrogen levels show a declining trend from the first week (approximately 30.75 mg/kg) to the third week (approximately 29.25 mg/kg), followed by a sharp recovery in the final week.
- This indicates possible depletion of nitrogen in the soil mid-month, potentially due to plant uptake, with replenishment or reduced usage in the last week.

#### **3. Phosphorus Level:**
- Phosphorus levels peak during the second week (approximately 31.5 mg/kg) and then decrease consistently through the remaining weeks (approximately 29.5 mg/kg by the end of the month).
- The rise and fall in phosphorus levels may indicate specific fertilization events followed by plant absorption or leaching.

#### **4. Potassium Level:**
- Potassium levels also follow a similar trend to phosphorus, starting high (approximately 31 mg/kg), dropping sharply mid-month (approximately 29.8 mg/kg), and recovering slightly in the last week.
- This pattern suggests a relationship between plant nutrient requirements and external replenishment.

# <span style="color:transparent;">Data Preprocessing</span>

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">Data Preprocessing</h1>
</div>

In [ ]:
# Custom mapping for Plant_Health_Status
custom_mapping = {'High Stress': 2, 'Moderate Stress': 1, 'Healthy': 0}
df['Plant_Health_Status_Encoded'] = df['Plant_Health_Status'].map(custom_mapping)

# Count unique values in Plant_Health_Status before encoding
unique_value_counts_before = df['Plant_Health_Status'].value_counts()

# Count unique values in Plant_Health_Status_Encoded after encoding
unique_value_counts_after = df['Plant_Health_Status_Encoded'].value_counts()

# Display unique values before and after encoding
print("----- Unique Values in Plant Health Status Before Encoding ----- ")
print(unique_value_counts_before)

print("\n----- Unique Values in Plant Health Status After Encoding ----- ")
print(unique_value_counts_after)

In [ ]:
numerical_features = df.select_dtypes(include=[np.number]).columns

# Z-score Method
z_scores = zscore(df[numerical_features])
outliers_zscore = (np.abs(z_scores) > 3).sum(axis=0)
print("Outliers Detected with Z-scores:\n", outliers_zscore)

### **Outliers Detected with Z-scores:**
- No outliers were detected across any of the numerical features in the dataset. 

This indicates that the numerical data is well within the expected range, with no extreme values exceeding the Z-score threshold of ±3.

In [ ]:
# Drop the original Plant_Health_Status column
df= df.drop(columns=['Plant_Health_Status', 'Week'])

In [ ]:
# Calculate correlations
correlations = df.corr()['Plant_Health_Status_Encoded'].sort_values(ascending=False)

# Convert to DataFrame
correlation_table = correlations.to_frame(name='Correlation with Plant_Health_Status_Encoded').reset_index()
correlation_table.rename(columns={'index': 'Feature'}, inplace=True)

# Display the table
display(correlation_table)


In [ ]:
# Ensure 'Plant_Health_Status_Encoded' is included and calculate mean values
status_correlation = df.groupby('Plant_Health_Status_Encoded').mean().T

plt.figure(figsize=(10, 8))
sns.heatmap(status_correlation, annot=True, cmap='YlGnBu_r', fmt='.2f', linewidths=0.5, square=True, cbar_kws={"shrink": 1})
plt.title('Feature Interaction with Plant Health Status (Encoded)')
plt.ylabel('Features')
plt.xlabel('Plant Health Status Encoded')
plt.tight_layout()
plt.show()

### Insights from the Feature Interaction with Plant_Health_Status_Encoded

The heatmap and the mean values table provide a clear view of how numerical features vary with different levels of plant health status (`0` = Healthy, `1` = Moderate Stress, `2` = High Stress).

### **1. Soil Properties:**
- **Soil Moisture:**
  - Healthy plants (`0`) have the highest average soil moisture (approximately 34.91%), indicating adequate hydration.
  - Moderate stress (`1`) plants exhibit reduced moisture (approximately 26.56%), while high-stress (`2`) plants have the lowest levels (approximately 18.08%), signifying severe water stress.

- **Soil pH:**
  - Healthy and moderate stress plants have a near-neutral pH (approximately 6.50), which is optimal for nutrient absorption.
  - High-stress plants show slightly elevated pH (approximately 6.55), which could reduce nutrient availability.

- **Nitrogen, Phosphorus, and Potassium Levels:**
  - Healthy plants exhibit the highest nutrient levels (Nitrogen: approximately 34.79 mg/kg, Phosphorus: approximately 31.40 mg/kg, Potassium: approximately 31.23 mg/kg), supporting growth and resilience.
  - Nutrient levels decrease progressively for moderate and high-stress plants, with high-stress plants showing the lowest levels (Nitrogen: approximately 26.97 mg/kg, Phosphorus: approximately 29.65 mg/kg, Potassium: approximately 30.27 mg/kg).

### **2. Environmental Conditions:**
- **Ambient Temperature and Soil Temperature:**
  - These values are relatively stable across health statuses (approximately 23.97°C for ambient and approximately 19.93°C for soil temperature), suggesting minimal impact of temperature on plant stress in this dataset.

- **Humidity:**
  - Healthy plants experience slightly higher humidity levels (approximately 55.36%) compared to high-stress plants (approximately 54.40%), indicating optimal transpiration conditions for healthy plants.

- **Light Intensity:**
  - Healthy plants are associated with slightly lower light intensity (approximately 603 Lux) compared to high-stress plants (approximately 618 Lux). Excessive light exposure could contribute to stress in plants.

### **3. Plant Health Indicators:**
- **Chlorophyll Content:**
  - Healthy plants show marginally higher chlorophyll levels (approximately 34.97 mg/m²) compared to moderate and high-stress plants (approximately 34.57 and approximately 34.76 mg/m², respectively).
  - The slight decline in chlorophyll content reflects reduced photosynthetic activity under stress conditions.

- **Electrochemical Signal:**
  - Healthy plants have the lowest electrochemical signal (approximately 0.965 mV), indicating minimal stress.
  - The signal increases progressively for moderate stress (approximately 0.992 mV) and high-stress plants (approximately 0.998 mV), reflecting heightened stress responses.

In [ ]:
# Define features and target
X = df.drop(columns=['Plant_Health_Status_Encoded'])  # Features
y = df['Plant_Health_Status_Encoded']  # Target


In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Display the shapes of the splits for verification
print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# <span style="color:transparent;">Model Training and Evaluation</span>

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">Model Training and Evaluation</h1>
</div>

In [ ]:
# Initialize models
models = {
    "KNN Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Display model names to confirm initialization
print("Models initialized:", list(models.keys()))

In [ ]:
# Initialize lists to store results
results = []

# Train and evaluate each model
for model_name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Calculate AUC
    if hasattr(model, "predict_proba"):  
        y_prob = model.predict_proba(X_test_scaled)
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    else:
        auc = None  
    
    # Store results
    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Confusion Matrix": confusion_matrix(y_test, y_pred),
        "Classification Report": classification_report(y_test, y_pred, target_names=['Healthy', 'Moderate Stress', 'High Stress'])
    })

# Display results for each model
for result in results:
    print(f"\nModel: {result['Model']}")
    print(f"Accuracy: {result['Accuracy']:.2f}")
    if result["AUC"] is not None:
        print(f"AUC: {result['AUC']:.2f}")
    
    # Confusion Matrix Visualization
    cm = result["Confusion Matrix"]
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', xticklabels=['Healthy', 'Moderate Stress', 'High Stress'], 
                yticklabels=['Healthy', 'Moderate Stress', 'High Stress'])
    plt.title(f'Confusion Matrix for {result["Model"]}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # Print Classification Report
    print(result["Classification Report"])


### **Key observations:**
1. **Tree-based Models Dominate:**
   - Decision Tree, Random Forest, and Gradient Boosting perform significantly better than KNN, achieving nearly perfect accuracy, AUC, and class-wise metrics.
   - Random Forest and Gradient Boosting are ideal for this dataset due to their ability to generalize well.

2. **KNN Classifier Struggles:**
   - Lower accuracy and class-wise metrics make KNN less reliable for this task, especially for distinguishing `Moderate Stress`.

3. **Best Model:**
   - **Random Forest** stands out as the best model due to its perfect AUC (1.00) and robust performance across all metrics.

In [ ]:
# Prepare a summary table for model evaluation
evaluation_summary = []

# Extract the key metrics for each model
for result in results:
    evaluation_summary.append({
        "Model": result["Model"],
        "Accuracy": result["Accuracy"],
        "AUC": result["AUC"]
    })

evaluation_summary_df = pd.DataFrame(evaluation_summary)

# Sort by accuracy and display
display(evaluation_summary_df.sort_values(by="Accuracy", ascending=False))


In [ ]:
# Select the Model with the Highest AUC
best_model_name = max(results, key=lambda x: x["AUC"] if x["AUC"] is not None else -1)["Model"]
best_model = models[best_model_name]
print(f"The best model based on AUC is: {best_model_name}")

In [ ]:
# Plot the Feature Importances
if hasattr(best_model, "feature_importances_"):  # Check if the model has feature_importances_
    feature_importances = best_model.feature_importances_
    feature_names = X.columns

    plt.figure(figsize=(10, 6))
    sorted_idx = feature_importances.argsort()[::-1]

    # Using color palette 'winter' for the bars
    colors = plt.cm.YlGnBu_r(np.linspace(0, 1, len(feature_importances)))

    plt.barh(range(len(sorted_idx)), feature_importances[sorted_idx], align='center', color=colors)
    plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
    plt.xlabel("Feature Importance")
    plt.title(f"Feature Importances for {best_model_name}")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not support feature importances.")


### Insights from Feature Importances for Random Forest:

The Random Forest model identifies the most critical features influencing the prediction of `Plant_Health_Status`. 

#### **Top Features:**
1. **Soil_Moisture (Most Important):**
   - Soil moisture contributes the highest importance (~60%) to the model's predictions.
   - This aligns with the understanding that water availability is crucial for plant health, as low soil moisture often leads to plant stress.

2. **Nitrogen_Level (Second Most Important):**
   - Nitrogen level contributes significantly to the model (~20%).
   - As a vital nutrient for plant growth, its deficiency directly impacts leaf development and overall health.

#### **Moderately Important Features:**
3. **Ambient_Temperature and Potassium_Level:**
   - These features have smaller but notable impacts on plant health.
   - **Ambient Temperature:** Affects plant stress indirectly by influencing transpiration and photosynthesis.
   - **Potassium Level:** Enhances disease resistance and water regulation in plants.

4. **Light_Intensity and Soil_pH:**
   - **Light Intensity:** Essential for photosynthesis; however, excessive light can cause stress.
   - **Soil pH:** Influences nutrient availability; a neutral pH is critical for healthy plant growth.

#### **Least Important Features:**
5. **Phosphorus_Level, Electrochemical_Signal, and Others:**
   - These features have lower contributions but still play a role in specific cases:
     - **Phosphorus_Level:** Important for root development but less variable in the dataset.
     - **Electrochemical_Signal:** Indicates stress but might overlap with other variables.
     - **Humidity and Soil Temperature:** Have minimal variation across health statuses.

6. **Plant_ID (Least Important):**
   - This is expected, as the unique identifier does not directly influence plant health.

### **Key observations:**
- **Primary Drivers of Plant Health:** Soil moisture and nitrogen levels are the most crucial factors for predicting plant health status.
- **Environmental Factors:** While factors like temperature and light intensity matter, they are secondary in importance compared to soil properties.
- **Data-Driven Insights:** Focus interventions (e.g., irrigation, fertilization) on maintaining optimal soil moisture and nutrient levels to improve plant health.

In [ ]:
# Generate Predictions
y_pred_best = best_model.predict(X_test_scaled)

# Plot the Distribution of Predictions
plt.figure(figsize=(8, 6))
sns.countplot(x=y_pred_best, palette='YlGnBu')
plt.xticks([0, 1, 2], labels=["Healthy", "Moderate Stress", "High Stress"])
plt.xlabel("Predicted Plant Health Status")
plt.ylabel("Count")
plt.title(f"Distribution of Predictions by {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

# Convert the actual and predicted values to a DataFrame for comparison
actual_vs_predicted = pd.DataFrame({
    'Actual': y_test,
    'Predicted': best_model.predict(X_test_scaled)
})

sns.countplot(
    data=actual_vs_predicted.melt(var_name='Type', value_name='Plant Health Status'),
    x='Plant Health Status', hue='Type', palette='YlGnBu'
)

plt.xticks([0, 1, 2], labels=["Healthy", "Moderate Stress", "High Stress"])
plt.xlabel("Plant Health Status")
plt.ylabel("Count")
plt.title("Actual vs Predicted Plant Health Status")
plt.legend(title="Type", loc="upper left")
plt.tight_layout()
plt.show()


### **Overall Model Performance:**
1. The model demonstrates strong alignment between actual and predicted values for all classes, particularly for "High Stress" and "Moderate Stress."
2. The small discrepancies in the "Healthy" and "Moderate Stress" categories suggest that these classes might have overlapping feature spaces, making them slightly harder to distinguish.

<div style="border-radius: 15px 0 15px 0px; border: 2px solid #f1c40f; padding: 10px; background-color: #055E68; text-align: center; box-shadow: 0px 2px 4px rgba(0, 0, 0, 0.2);">
    <h1 style="color: #f1c40f; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.5); font-weight: bold; margin-bottom: 10px; font-size: 24px;">🚀 Enjoying this notebook? Consider giving it an upvote! 👍</h1>
    <p style="color: #ecf0f1; font-size: 18px; text-align: center;">Your support inspires me to create more great content and helps others find it too! 🙌</p>
    <p style="color: #ecf0f1; font-size: 18px; text-align: center;">Thanks for stopping by happy data diving! 🌈😊</p>
</div>
